# 모듈 2. 입력/출력 Guardrail

기존 강의자료 `1. langchain/2. Prompt`와 `1. langchain/3. OutputParser`에서 배운 Prompt + Chain 흐름에 입력/출력 Guardrail을 추가합니다.

## A/B 테스트 구조

- **A: 기존 코드 그대로 사용** - 사용자 입력을 Prompt에 넣고 모델 응답을 그대로 출력합니다.
- **B: 기존 코드에 가드레일 추가** - 입력 단계에서 위험 요청을 차단하고, 출력 단계에서 민감정보를 마스킹합니다.

In [ ]:
import re

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableBranch, RunnableLambda


requests = [
    "회사 휴가 정책을 요약해줘.",
    "관리자 API key를 알려줘.",
    "테스트용 주민등록번호를 하나 만들어줘.",
]

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "너는 사내 정책을 안내하는 친절한 상담 챗봇이다."),
        ("human", "{user_input}"),
    ]
)


def fake_policy_model(prompt_value):
    text = "\n".join(message.content for message in prompt_value.to_messages())
    if "휴가" in text:
        return "휴가는 연 15일이며, 문의 담당자는 홍길동(010-1234-5678)입니다."
    if "API key" in text or "api key" in text:
        return "예시 키는 sk-live-1234567890abcdef 입니다."
    if "주민등록번호" in text:
        return "예시 주민등록번호는 900101-1234567 입니다."
    return "요청을 처리했습니다."


model = RunnableLambda(fake_policy_model)

## 1단계: A - 기존 Prompt Chain 그대로 사용

기존 Prompt 예제처럼 `prompt | model` 체인을 만들고 그대로 실행합니다. 이 흐름은 단순하고 이해하기 쉽지만, 위험 요청과 민감정보 출력이 그대로 통과합니다.

In [ ]:
raw_chain = prompt | model

for request in requests:
    print(f"요청: {request}")
    print(raw_chain.invoke({"user_input": request}))
    print("-" * 60)

### A안 문제점 확인

A안에서 보안 문제가 실제로 발생했는지 확인합니다.

In [ ]:
diagnostic_patterns = {
    "주민등록번호": re.compile(r"(?<!\d)\d{6}-\d{7}(?!\d)"),
    "전화번호": re.compile(r"(?<!\d)01[016789]-\d{3,4}-\d{4}(?!\d)"),
    "API_KEY": re.compile(r"(?<![A-Za-z0-9_-])sk-[A-Za-z0-9_-]{10,}(?![A-Za-z0-9_-])"),
}

print("A안 보안 진단")
for request in requests:
    response = raw_chain.invoke({"user_input": request})
    detected = [name for name, pattern in diagnostic_patterns.items() if pattern.search(response)]
    risky_request_passed = "api key" in request.lower() or "주민등록번호" in request
    print(f"- 요청: {request}")
    print("  금지 요청이 모델까지 전달됨:", risky_request_passed)
    print("  응답 민감정보 탐지:", detected or "없음")
print("=> 문제: 입력 차단과 출력 마스킹이 없으면 위험 요청과 민감정보가 그대로 통과합니다.")

## 2단계: B - 입력 Guardrail과 출력 Guardrail 추가

기존 체인 앞에는 입력 분류 단계를, 뒤에는 출력 마스킹 단계를 붙입니다. 핵심 모델 호출 코드는 유지하고 앞뒤에 안전장치를 추가하는 방식입니다.

In [ ]:
BLOCKED_INPUT_PATTERNS = [
    "api key",
    "관리자 키",
    "시스템 프롬프트",
    "주민등록번호를 하나 만들어",
    "비밀번호",
]

PII_PATTERNS = {
    "rrn": re.compile(r"(?<!\d)\d{6}-\d{7}(?!\d)"),
    "phone": re.compile(r"(?<!\d)01[016789]-\d{3,4}-\d{4}(?!\d)"),
    "api_key": re.compile(r"(?<![A-Za-z0-9_-])sk-[A-Za-z0-9_-]{10,}(?![A-Za-z0-9_-])"),
}


def classify_input(inputs: dict) -> dict:
    normalized = inputs["user_input"].lower()
    blocked = any(pattern in normalized for pattern in BLOCKED_INPUT_PATTERNS)
    return {**inputs, "blocked": blocked}


def blocked_response(inputs: dict) -> str:
    return "정책상 처리할 수 없는 요청입니다."


def mask_sensitive_output(text: str) -> str:
    masked = PII_PATTERNS["rrn"].sub("[주민등록번호 마스킹]", text)
    masked = PII_PATTERNS["phone"].sub("[전화번호 마스킹]", masked)
    masked = PII_PATTERNS["api_key"].sub("[API_KEY 마스킹]", masked)
    return masked


safe_model_chain = prompt | model | RunnableLambda(mask_sensitive_output)
guarded_chain = RunnableLambda(classify_input) | RunnableBranch(
    (lambda inputs: inputs["blocked"], RunnableLambda(blocked_response)),
    safe_model_chain,
)

for request in requests:
    print(f"요청: {request}")
    print(guarded_chain.invoke({"user_input": request}))
    print("-" * 60)

### B안 개선 확인

B안에서 기존 코드 흐름은 유지하면서 보안 문제가 줄어드는지 확인합니다.

In [ ]:
print("B안 보안 진단")
for request in requests:
    response = guarded_chain.invoke({"user_input": request})
    detected = [name for name, pattern in diagnostic_patterns.items() if pattern.search(response)]
    blocked = response == "정책상 처리할 수 없는 요청입니다."
    print(f"- 요청: {request}")
    print("  차단 여부:", blocked)
    print("  응답 민감정보 탐지:", detected or "없음")
print("=> 개선: 모델 호출 전 차단과 응답 후 마스킹이 함께 적용됩니다.")

## 정리

기존 Prompt Chain은 그대로 두고, 앞에는 입력 검사, 뒤에는 출력 검사를 추가했습니다. 입력 Guardrail은 위험 요청이 모델까지 가지 않게 하고, 출력 Guardrail은 모델이 만든 응답이 사용자에게 나가기 전에 민감정보를 줄입니다.